# TRANSFORMATION DES DONNÉES TEXTUELES EN DONNÉES EXPLOITABLES PAR UN MODÈLE

## Importation des dépendances nécessaires

In [21]:
import math
import pandas as pd
import numpy as np
import string
from collections import Counter
import joblib

## Chargement du cleaned data issue du préprocessing initial

In [22]:
df = pd.read_csv("../data/cleaned_data.csv", encoding="latin-1")
print(df.head(10))

     v1                                                 v2
0   ham  Yes I know the cheesy songs from frosty the sn...
1  spam  FREEMSG: Our records indicate you may be entit...
2   ham                                               Okie
3   ham  Quite late lar... Ard 12 anyway i wun b drivin...
4   ham                          Those ducking chinchillas
5   ham  Nowadays people are notixiquating the laxinorf...
6  spam  Hi 07734396839 IBH Customer Loyalty Offer: The...
7   ham          It's ok lar. U sleep early too... Nite...
8  spam  Ur cash-balance is currently 500 pounds - to m...
9   ham  Ok that would b lovely, if u r sure. Think abo...


## Netoyage du texte

### Transformer tout le texte en minuscule (lowercase)

In [23]:
# tranformation colone par colone
for key in df.keys():
    df[key] = df[key].str.lower()

# Verification de la transformation
print(df)

        v1                                                 v2
0      ham  yes i know the cheesy songs from frosty the sn...
1     spam  freemsg: our records indicate you may be entit...
2      ham                                               okie
3      ham  quite late lar... ard 12 anyway i wun b drivin...
4      ham                          those ducking chinchillas
...    ...                                                ...
1301   ham  gud ni8 dear..slp well..take care..swt dreams....
1302   ham  ah poop. looks like ill prob have to send in m...
1303   ham                   you didnt complete your gist oh.
1304   ham         i'll text carlos and let you know, hang on
1305   ham  get down in gandhipuram and walk to cross cut ...

[1306 rows x 2 columns]


### Supprimer la ponctuation

In [24]:
def remove_ponctuation(texte):
    if isinstance(texte, str):
        return texte.translate(str.maketrans('', '', string.punctuation))
    return texte

for key in df.keys():
    df[key] = df[key].apply(remove_ponctuation)


In [25]:
print(df.head(10))

     v1                                                 v2
0   ham  yes i know the cheesy songs from frosty the sn...
1  spam  freemsg our records indicate you may be entitl...
2   ham                                               okie
3   ham        quite late lar ard 12 anyway i wun b drivin
4   ham                          those ducking chinchillas
5   ham  nowadays people are notixiquating the laxinorf...
6  spam  hi 07734396839 ibh customer loyalty offer the ...
7   ham                  its ok lar u sleep early too nite
8  spam  ur cashbalance is currently 500 pounds  to max...
9   ham  ok that would b lovely if u r sure think about...


## TOKENISATION

La tokenisation consiste à décomposer un texte brut (une phrase, un paragraphe, un document) en unités plus petites et gérables, appelées tokens (ou jetons). 

### Pourquoi la tokenisation est-elle indispensable ?
- Conversion texte-nombre : Les modèles de ML sont des modèles mathématiques ; ils ne comprennent pas le langage humain brut.
- Gestion du vocabulaire : Elle définit le "vocabulaire" que le modèle peut comprendre.
- Performance et contexte : Elle impacte la vitesse du modèle et sa capacité à traiter de longs textes (limite de contexte).
- Coûts : Dans les LLM, la tokenisation détermine le coût d'utilisation de l'API (facturation au token).

Ici on utilise un simple split sans exploiter les méthodes avancées puisqu'on fait otut from scratch

In [26]:

df["tokens"] = df["v2"].str.split()

df

,v1,v2,tokens
0,ham,yes i know the cheesy songs from frosty the sn...,"[yes, i, know, the, cheesy, songs, from, frost..."
1,spam,freemsg our records indicate you may be entitl...,"[freemsg, our, records, indicate, you, may, be..."
2,ham,okie,[okie]
3,ham,quite late lar ard 12 anyway i wun b drivin,"[quite, late, lar, ard, 12, anyway, i, wun, b,..."
4,ham,those ducking chinchillas,"[those, ducking, chinchillas]"
...,...,...,...
1301,ham,gud ni8 dearslp welltake careswt dreamsmuah,"[gud, ni8, dearslp, welltake, careswt, dreamsm..."
1302,ham,ah poop looks like ill prob have to send in my...,"[ah, poop, looks, like, ill, prob, have, to, s..."
1303,ham,you didnt complete your gist oh,"[you, didnt, complete, your, gist, oh]"
1304,ham,ill text carlos and let you know hang on,"[ill, text, carlos, and, let, you, know, hang,..."


## Tri sélectif (Stop words)

Le tri sélectif  consiste à éliminer les mots qui apparaissent très fréquemment dans une langue (comme "le", "la", "et", "de", "un") mais qui apportent très peu de valeur informative pour la compréhension du sens global d'une phrase.

### Pourquoi le tri sélectif?

- Réduire la dimensionnalité : Moins de mots uniques dans le vocabulaire signifie des modèles plus légers et plus rapides à entraîner.
- Améliorer la pertinence : En supprimant le "bruit" (les mots non significatifs), le modèle se concentre sur les mots porteurs de sens (ex: "banane", "rouge", "voiture").
- Optimiser la performance : Cela permet d'augmenter la précision des algorithmes de classification de texte, de recherche d'information ou d'analyse de sentiments.



In [27]:
# fonction pour suprimer les stops words

def remove_stops_words(list, stop_words):
    new_list = []
    for word in list:
        if not word in stop_words:
            new_list.append(word)
    return new_list

# list des stop words fréquent en anglais

stop_words = [
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've",
    "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself',
    'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them',
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll",
    'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has',
    'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or',
    'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against',
    'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from',
    'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once',
    'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
    'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than',
    'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should', "should've",
    'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't",
    'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven',
    "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn',
    "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren',
    "weren't", 'won', "won't", 'wouldn', "wouldn't"]

# application à notre dataframe

df['tokens'] = df['tokens'].apply(lambda x: remove_stops_words(x, stop_words))

df

,v1,v2,tokens
0,ham,yes i know the cheesy songs from frosty the sn...,"[yes, know, cheesy, songs, frosty, snowman]"
1,spam,freemsg our records indicate you may be entitl...,"[freemsg, records, indicate, may, entitled, 37..."
2,ham,okie,[okie]
3,ham,quite late lar ard 12 anyway i wun b drivin,"[quite, late, lar, ard, 12, anyway, wun, b, dr..."
4,ham,those ducking chinchillas,"[ducking, chinchillas]"
...,...,...,...
1301,ham,gud ni8 dearslp welltake careswt dreamsmuah,"[gud, ni8, dearslp, welltake, careswt, dreamsm..."
1302,ham,ah poop looks like ill prob have to send in my...,"[ah, poop, looks, like, ill, prob, send, lapto..."
1303,ham,you didnt complete your gist oh,"[didnt, complete, gist, oh]"
1304,ham,ill text carlos and let you know hang on,"[ill, text, carlos, let, know, hang]"


## Retour à la racine des mots (Stemming ou racinisation)

C'est une technique de prétraitement de texte qui consiste à réduire un mot à sa forme de base ou radical, en supprimant les suffixes et préfixes, afin de normaliser le vocabulaire.
L'objectif principal est de regrouper différentes formes fléchies d'un même mot sous une seule forme unique pour simplifier l'analyse. 

### Pourquoi la racinisation ?

La racinisation (ou stemming) est utilisée en Machine Learning et en NLP principalement pour simplifier et normaliser le texte avant son analyse par un algorithme.

- Réduction de la dimensionnalité : En ramenant des mots comme "jouer", "joueur" et "jouons" à la même racine ("jou"), on réduit considérablement le nombre de mots uniques (le vocabulaire) que le modèle doit apprendre. Cela accélère l'entraînement et réduit la mémoire nécessaire.
- Regroupement sémantique : Elle permet de traiter différentes formes d'un même concept comme une seule entité. Pour un moteur de recherche, cela garantit que si vous cherchez "pêcheur", vous trouverez aussi des documents contenant "pêcher".
- Amélioration de la performance : En regroupant les variantes, on renforce le "signal" statistique pour chaque concept, ce qui peut rendre les modèles de classification de texte ou d'analyse de sentiment plus robustes.
- Rapidité et simplicité : Contrairement à la lemmatisation (qui cherche le mot exact dans un dictionnaire), la racinisation utilise des règles de découpe simples. Elle est donc beaucoup plus rapide pour traiter de très gros volumes de données

In [28]:
# fonction principale

def stemmer_simple(word):
    word = word.lower()
    
    # Suffixes anglais courants
    suffixes = ['ing', 'ly', 'ed', 'ies', 'es', 's', 'ment', 'able', 'ness']
    
    # Cas particuliers simples (ex: "flies" -> "fly")
    if word.endswith('ies'):
        return word[:-3] + 'y'
    
    for suffix in suffixes:
        if word.endswith(suffix):
            # On garde une racine de 3 lettres minimum
            if len(word) - len(suffix) >= 3:
                return word[:-len(suffix)]
    return word


# application à notre dataframe

# On garde la structure de liste par ligne
df["tokens"] = [[stemmer_simple(word) for word in liste_mots] for liste_mots in df["tokens"]]


## LA VECTORISATION (Transformation en Nombres)



La vectorisation en Machine Learning (ML) est le processus qui consiste à convertir des données brutes (textes, images, sons) en listes de nombres (vecteurs). 

### POURQUOI?

Elle est indispensable pour deux raisons majeures :

- **Rendre les données compréhensibles par la machine :** Les algorithmes de ML ne comprennent pas le langage humain ou les pixels. La vectorisation traduit ces informations en coordonnées mathématiques dans un espace multidimensionnel.

- **Effectuer des calculs ultra-rapides :** En informatique, manipuler des matrices et des vecteurs permet d'effectuer des opérations en parallèle au lieu de traiter les données une par une.

## TF-IDF (Term Frequency - Inverse Document Frequency)

Contrairement au simple comptage de mots (Bag of Words), le TF-IDF permet de mesurer l'importance réelle d'un terme dans un message par rapport à l'ensemble du dataset.

### Pourquoi l'utiliser pour la détection de spams ?

**Élimination du bruit :** Les mots très fréquents mais peu informatifs (ex: "le", "de", "votre") voient leur importance réduite automatiquement.

**Mise en valeur des mots-clés :** Les termes caractéristiques des spams (ex: "gagné", "cliquez", "promotion") obtiennent un score plus élevé car ils sont fréquents dans certains messages spécifiques mais rares dans les échanges normaux.

### Formule simplifiée :

$Score=TF\times IDF$

**TF (Term Frequency) :** Fréquence du mot dans le message actuel.

**IDF (Inverse Document Frequency) :** Rareté du mot sur l'ensemble de tous les messages.

In [29]:
# fonction pour calculer le Term Frequency (TF)

def calculate_tf(tokens):
    total_words = len(tokens)
    tf_dict = {}

    # compte des occurences
    frequences = Counter(tokens)
    for word, count in frequences.items():
        tf_dict[word] = count / total_words
    
    return tf_dict

# fonction pour calculer le Inverse Document Frequency (IDF)
def calculate_idf(corpus_tokens):
    total_count = len(corpus_tokens)
    idf_dict = {}
    
    # Ensemble de tous les mots uniques dans l'ensemble du corpus
    unique_words = set(word for doc in corpus_tokens for word in doc)
    
    for word in unique_words:
        # Compte combien de documents contiennent ce mot
        word_docs_sum = sum(1 for doc in corpus_tokens if word in set(doc))
        
        # Formule IDF avec lissage (smooth) pour éviter les divisions par zéro
        idf_dict[word] = math.log((1 + total_count) / (1 + word_docs_sum)) + 1
        
    return idf_dict

# fonction pour calculer le tf-idf

def calculate_tfidf(corpus_tokens):
    # Calcul du TF pour chaque document
    tf_corpus = [calculate_tf(doc) for doc in corpus_tokens]
    
    # Calcul du score IDF global du corpus
    idf_corpus = calculate_idf(corpus_tokens)
    
    tfidf_corpus = []
    for tf_doc in tf_corpus:
        tfidf_doc = {}
        for mot, valeur_tf in tf_doc.items():
            # TF-IDF = TF * IDF
            tfidf_doc[mot] = valeur_tf * idf_corpus[mot]
        tfidf_corpus.append(tfidf_doc)
        
    return tfidf_corpus

# UTILISATION SUR NOTRE DATAFRAME

result_tfidf = calculate_tfidf(df['tokens'])

# Affichage du résultat
for i, doc_tfidf in enumerate(result_tfidf):
    print(f"Document {i+1} :")
    for mot, score in doc_tfidf.items():
        print(f"  - {mot} : {score:.4f}")

Document 1 :
  - yes : 0.8474
  - know : 0.7357
  - cheesy : 1.2471
  - song : 1.0640
  - frosty : 1.2471
  - snowman : 1.2471
Document 2 :
  - freemsg : 0.3507
  - record : 0.3893
  - indicate : 0.4423
  - may : 0.3557
  - entitl : 0.3893
  - 3750 : 0.4676
  - pound : 0.3178
  - accident : 0.4423
  - claim : 0.2270
  - free : 0.1954
  - rep : 0.2318
  - yes : 0.3178
  - msg : 0.2870
  - opt : 0.3557
  - text : 0.2112
  - stop : 0.2284
Document 3 :
  - okie : 7.0769
Document 4 :
  - quite : 0.6323
  - late : 0.6773
  - lar : 0.6922
  - ard : 0.6922
  - 12 : 0.6643
  - anyway : 0.7296
  - wun : 0.8314
  - b : 0.5507
  - drivin : 0.7863
Document 5 :
  - duck : 3.7412
  - chinchilla : 3.7412
Document 6 :
  - nowaday : 0.3401
  - people : 0.2670
  - notixiquat : 0.3401
  - laxinorficat : 0.3401
  - opportunity : 0.3401
  - bambl : 0.3401
  - entropication : 0.3401
  - ever : 0.2717
  - oblisingate : 0.3401
  - opt : 0.2587
  - ur : 0.3103
  - book : 0.2626
  - masteriaster : 0.3401
  - amp

## Transformation des dictionnaires TF-IDF en Matrice Numérique Finale

Actuellement, les scores TF-IDF de notre corpus sont stockés sous forme d'une liste de dictionnaires (un dictionnaire par message). Pour qu'un algorithme de Machine Learning puisse traiter ces données, nous devons standardiser ce format.

L'objectif de cette étape est de :
1. **Extraire le vocabulaire unique global** : Identifier tous les mots distincts conservés à travers l'ensemble des messages.
2. **Construire la matrice de caractéristiques $X$** : Créer un tableau à deux dimensions de taille `(Nombre de documents × Taille du vocabulaire)`. Chaque ligne représentera un message et chaque colonne correspondra à un mot unique. Si un mot est absent d'un document, son score sera de `0`.
3. **Encoder la variable cible $y$** : Convertir les étiquettes textuelles (`ham` et `spam`) en valeurs numériques binaires (`0` et `1`).
4. **Sauvegarder les structures** : Exporter les matrices au format binaire NumPy (`.npy`) pour pouvoir les charger instantanément dans l'étape de modélisation sans refaire les calculs.

In [30]:
# Extraction du vocabulaire unique global
global_vocabulary = set()
for document_tfidf in result_tfidf:
    global_vocabulary.update(document_tfidf.keys())

# Tri du vocabulaire pour garantir un ordre constant des colonnes
global_vocabulary = sorted(list(global_vocabulary))
vocabulary_size = len(global_vocabulary)
total_documents = len(result_tfidf)

print(f"Nombre total de messages : {total_documents}")
print(f"Nombre de mots uniques (Taille du vocabulaire) : {vocabulary_size}")

# Construction de la matrice de caractéristiques X (Documents x Mots)
# On initialise une matrice remplie de zéros avec NumPy
X_features = np.zeros((total_documents, vocabulary_size))

# Création d'un dictionnaire d'index pour retrouver instantanément la colonne d'un mot
word_to_index = {word: index for index, word in enumerate(global_vocabulary)}

# Remplissage de la matrice X avec les scores TF-IDF correspondants
for doc_index, document_tfidf in enumerate(result_tfidf):
    for word, tfidf_score in document_tfidf.items():
        if word in word_to_index:
            target_column = word_to_index[word]
            X_features[doc_index, target_column] = tfidf_score

# Encodage binaire de la variable cible y (ham = 0, spam = 1)
y_labels = df['v1'].map({'ham': 0, 'spam': 1}).to_numpy()

# Sauvegarde des fichiers au format binaire NumPy (.npy)
np.save('../src/X_features.npy', X_features)
np.save('../src/y_labels.npy', y_labels)

idf_dict_propre = calculate_idf(df['tokens'])
idf_dict_trie = {word: idf_dict_propre[word] for word in sorted(idf_dict_propre.keys())}
# On sauvegarde le dictionnaire trié
joblib.dump(idf_dict_trie, '../src/idf_corpus.pkl')

print("La matrice X_features et le vecteur y_labels ont été sauvegardés avec succès !")

Nombre total de messages : 1306
Nombre de mots uniques (Taille du vocabulaire) : 4185
La matrice X_features et le vecteur y_labels ont été sauvegardés avec succès !
